# Тема 2.2. Основы проектирования: Принципы, архитектура, стили

## 🎯 Цели занятия

После этого занятия вы сможете:
1.  Объяснять ключевые принципы проектирования ПО (SOLID, DRY, KISS, YAGNI) и применять их на практике.
2.  Понимать, что такое архитектура ПО, зачем она нужна и из каких аспектов состоит.
3.  Описывать историю возникновения и проблемы, которые решают основные архитектурные стили: монолит, N-уровневая, микросервисы, SOA, событийная.
4.  Связывать нефункциональные требования (NFR) с выбором архитектурного стиля.
5.  Сравнивать архитектурные стили по значимым критериям и выбирать подходящий для конкретной задачи.
6.  Применять полученные знания в практических заданиях.

---

## 📚 Часть 1. Принципы проектирования ПО

### 1.1. Зачем нужны принципы проектирования?

Принципы проектирования — это не догмы, а **эвристики**, помогающие создавать код, который легко читать, изменять и тестировать. Они возникли как ответ на реальные проблемы: «код, который невозможно поддерживать», «изменения в одном месте ломают три других», «новая функция требует переписывания половины системы».

### 1.2. SOLID

**История**: Акроним SOLID предложен Майклом Фезерсом (Michael Feathers) около 2004 года на основе принципов, сформулированных Робертом Мартином (Robert C. Martin, «Uncle Bob») в его статье 2000 года «Design Principles and Design Patterns». Цель — сделать ООП-дизайн устойчивым к изменениям.

#### S — Single Responsibility Principle (SRP)

**Принцип**: У класса должна быть только одна причина для изменения. Другими словами, класс должен решать одну задачу.

**Проблема**: Класс, который составляет и печатает отчёт, имеет две причины для изменения: изменение содержания отчёта и изменение формата печати. Это делает код хрупким.

**Пример (нарушение)**:
```python
class Report:
    def generate(self):
        # логика генерации
        pass
    def print(self):
        # логика печати
        pass
```

**Решение**: Разделить на `ReportGenerator` и `ReportPrinter`.

#### O — Open/Closed Principle (OCP)

**Принцип**: Программные сущности (классы, модули, функции) должны быть **открыты для расширения**, но **закрыты для модификации**.

**Проблема**: Функция `draw_all_shapes`, которая проверяет тип каждой фигуры через `if`, не соответствует OCP, потому что для добавления новой фигуры её придётся модифицировать.

**Пример (решение через абстракцию)**:
```python
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def draw(self): ...

class Circle(Shape):
    def draw(self):
        print("Circle")

class Square(Shape):
    def draw(self):
        print("Square")

def draw_all(shapes: list[Shape]):
    for shape in shapes:
        shape.draw()
```
Теперь для добавления новой фигуры не нужно менять `draw_all`.

#### L — Liskov Substitution Principle (LSP)

**Принцип**: Объекты подкласса должны быть заменяемы объектами базового класса без нарушения работы программы.

**Проблема**: Классический пример — `Square` наследует `Rectangle`. Если у `Rectangle` есть методы `set_width` и `set_height`, то для квадрата изменение ширины должно менять и высоту. Это нарушает ожидания клиента, который работает с `Rectangle`.

**Пример (нарушение)**:
```python
class Rectangle:
    def set_width(self, w): self.width = w
    def set_height(self, h): self.height = h

class Square(Rectangle):
    def set_width(self, w):
        self.width = self.height = w
    def set_height(self, h):
        self.width = self.height = h
```
Клиентский код, который ожидает независимого изменения сторон, сломается.

#### I — Interface Segregation Principle (ISP)

**Принцип**: Клиенты не должны зависеть от интерфейсов, которые они не используют.

**Проблема**: Интерфейс `Worker` с методами `work()`, `eat()`, `sleep()` заставляет класс `Robot` реализовывать `eat()` и `sleep()`, которые ему не нужны. Это «загрязняет» класс и вводит в заблуждение.

**Решение (через `typing.Protocol`)**: В Python для разделения интерфейсов удобно использовать структурную типизацию — `Protocol`. Класс не обязан явно наследоваться: достаточно иметь нужные методы.

```python
from typing import Protocol

class Workable(Protocol):
    def work(self) -> None: ...

class Eatable(Protocol):
    def eat(self) -> None: ...

class Sleepable(Protocol):
    def sleep(self) -> None: ...

class Human:
    def work(self) -> None: print("Working...")
    def eat(self) -> None: print("Eating...")
    def sleep(self) -> None: print("Sleeping...")

class Robot:
    def work(self) -> None: print("Working 24/7...")

def do_work(worker: Workable) -> None:
    worker.work()

def have_lunch(person: Eatable) -> None:
    person.eat()

do_work(Human())    # ✅
do_work(Robot())    # ✅
have_lunch(Human()) # ✅
# have_lunch(Robot()) # ❌ mypy: Robot has no attribute "eat"
```

**Альтернатива через `abc.ABC`**: Если нужен явный контракт и проверка через `isinstance()`:

```python
from abc import ABC, abstractmethod

class Workable(ABC):
    @abstractmethod
    def work(self) -> None: ...

class Eatable(ABC):
    @abstractmethod
    def eat(self) -> None: ...

class Robot(Workable):  # только Workable
    def work(self): print("Working 24/7...")
```

**Что выбрать**:
- `Protocol` — современный Pythonic-подход, работает со сторонними классами, проверяется mypy/pyright.
- `ABC` — явный контракт, проверка в рантайме, привычно для Java/C#-разработчиков.

#### D — Dependency Inversion Principle (DIP)

**Принцип**:
1. Модули верхнего уровня не должны зависеть от модулей нижнего уровня. Оба должны зависеть от **абстракций**.
2. Абстракции не должны зависеть от деталей. Детали должны зависеть от абстракций.

**Проблема**: Класс `Button` (высокоуровневая логика — «что делать при нажатии») напрямую зависит от класса `Lamp` (низкоуровневая деталь — «как включить лампу»). Если завтра понадобится управлять не лампой, а телевизором или вентилятором, придётся **менять код `Button`** — а это нарушает OCP и делает систему хрупкой.

**Пример (нарушение)**:
```python
class Lamp:
    def turn_on(self) -> None:
        print("Lamp: on")
    def turn_off(self) -> None:
        print("Lamp: off")

class Button:
    def __init__(self, lamp: Lamp):   # ❌ жёсткая зависимость от конкретного класса
        self.lamp = lamp
        self.is_on = False

    def press(self) -> None:
        if self.is_on:
            self.lamp.turn_off()
            self.is_on = False
        else:
            self.lamp.turn_on()
            self.is_on = True

button = Button(Lamp())
button.press()   # Lamp: on
```

**Что не так**:
- Чтобы добавить `TV`, придётся либо изменить `Button`, либо создать `TVButton` — дублирование.
- `Button` невозможно протестировать без реального `Lamp`.
- Высокоуровневая политика («переключать устройства») зависит от низкоуровневой детали («лампа»).

**Решение (через `typing.Protocol`)** — вводим абстракцию `SwitchableDevice`, от которой зависят оба модуля:

```python
from typing import Protocol

# Абстракция: "что-то, что можно включить/выключить"
class SwitchableDevice(Protocol):
    def turn_on(self) -> None: ...
    def turn_off(self) -> None: ...

# Низкоуровневые детали реализуют абстракцию
class Lamp:
    def turn_on(self) -> None: print("Lamp: on")
    def turn_off(self) -> None: print("Lamp: off")

class TV:
    def turn_on(self) -> None: print("TV: on")
    def turn_off(self) -> None: print("TV: off")

class Fan:
    def turn_on(self) -> None: print("Fan: on")
    def turn_off(self) -> None: print("Fan: off")

# Высокоуровневый модуль зависит только от абстракции
class Button:
    def __init__(self, device: SwitchableDevice):   # ✅ зависимость от абстракции
        self.device = device
        self.is_on = False

    def press(self) -> None:
        if self.is_on:
            self.device.turn_off()
            self.is_on = False
        else:
            self.device.turn_on()
            self.is_on = True

Button(Lamp()).press()   # Lamp: on
Button(TV()).press()     # TV: on
Button(Fan()).press()    # Fan: on
```

**Альтернатива через `abc.ABC`** — если нужен явный контракт:

```python
from abc import ABC, abstractmethod

class SwitchableDevice(ABC):
    @abstractmethod
    def turn_on(self) -> None: ...
    @abstractmethod
    def turn_off(self) -> None: ...

class Lamp(SwitchableDevice):
    def turn_on(self): print("Lamp: on")
    def turn_off(self): print("Lamp: off")
```

**Что даёт DIP**:
- **Расширяемость (OCP)**: новое устройство добавляется без правки `Button`.
- **Тестируемость**: в тестах можно передать `MockDevice`, не поднимая реальное железо.

```python
class MockDevice:
    def __init__(self):
        self.calls = []
    def turn_on(self): self.calls.append("on")
    def turn_off(self): self.calls.append("off")

mock = MockDevice()
btn = Button(mock)
btn.press()
btn.press()
assert mock.calls == ["on", "off"]   # ✅ тест без реальной лампы
```

- **Инверсия зависимостей**: раньше `Button` → `Lamp` (сверху вниз). Теперь и `Button`, и `Lamp` зависят от `SwitchableDevice`.

> ⚠️ **Важно не путать**: **DIP** — это принцип проектирования («завись от абстракций»). **Dependency Injection (DI)** — это техника реализации (передача зависимости извне, например, через конструктор). DI — один из способов соблюсти DIP, но не единственный (ещё есть Service Locator, фабрики и т.д.).

> 💡 **Правило большого пальца**: если в коде высокоуровневого модуля вы пишете `from myproject.database.postgres import PostgresConnection` — это почти наверняка нарушение DIP.

### 1.3. DRY, KISS, YAGNI

**История**:
- **DRY** (Don't Repeat Yourself) — сформулирован Эндрю Хантом и Дэвидом Томасом в книге «The Pragmatic Programmer» (1999).
- **KISS** (Keep It Simple, Stupid) — восходит к дизайн-этосу ВМС США 1960-х, популяризирован в ПО позже.
- **YAGNI** (You Aren't Gonna Need It) — пришёл из Extreme Programming (Кент Бек, конец 1990-х).

#### DRY

**Точное определение**: «Каждая часть знания должна иметь единственное, однозначное, авторитетное представление в системе».

**Важно**: DRY — это про **знание**, а не про символы. Два одинаковых блока кода, которые кодируют разные решения, не являются нарушением DRY. Их объединение — самый частый способ создать неправильную абстракцию.

**Пример (нарушение)**:
```python
def create_user(email: str):
    if not email or "@" not in email:
        raise ValueError("Invalid email")
    # ...

def update_email(user, new_email: str):
    if not new_email or "@" not in new_email:
        raise ValueError("Invalid email")
    # ...
```
**Решение**: Вынести валидацию в отдельную функцию.

#### KISS

**Принцип**: Простота — высшая форма sophistication. Код должен быть настолько простым, насколько это возможно, но не проще.

**Проблема**: Чрезмерно «умный» код (например, сложные однострочники с вложенными тернарными операторами) нечитаем и хрупок.

**Пример (нарушение)**:
```python
def is_valid_ip(ip):
    return bool(ip) and all(0 <= int(p) <= 255 for p in ip.split('.')) if len(ip.split('.')) == 4 else False
```
**Решение**: Разбить на понятные шаги с проверками.

#### YAGNI

**Принцип**: Не добавляйте функциональность, пока она не понадобится. Не «закладывайтесь на будущее» без конкретного требования сегодня.

**Проблема**: Кто-то «на будущее» добавил `QueueBrokerStrategyFactoryProvider` и три слоя интерфейсов для Kafka, Redis и RabbitMQ, которые никто не просил. Теперь добавление одного поля требует правки девяти файлов.

**Пример (нарушение)**:
```python
# На этапе MVP, когда нужен только один брокер
class MessageBrokerFactory:
    def create_broker(self, type: str):
        if type == "kafka": ...
        elif type == "rabbit": ...
        elif type == "redis": ...
```
**Решение**: Пока нужен только один брокер — используйте его напрямую.

> **Ключевая мысль**: DRY, KISS и YAGNI — это не независимые правила, а **напряжение**, которое нужно удерживать. Агрессивный DRY порождает преждевременную абстракцию. Агрессивная простота порождает дублирование.

---

## 📚 Часть 2. Создание архитектуры ПО

### 2.1. Что такое архитектура ПО?

Архитектура ПО — это **набор ключевых решений** о структуре системы: какие компоненты в неё входят, как они взаимодействуют, какие ограничения на них накладываются. Это «чертёж», определяющий, как система будет удовлетворять требованиям (функциональным и нефункциональным).

### 2.2. Зачем нужна архитектура?

Без архитектуры система превращается в «большой комок грязи» (Big Ball of Mud). Архитектура помогает:
- **Управлять сложностью**: разбивать систему на части, которые можно понимать и разрабатывать независимо.
- **Обеспечивать качество**: производительность, масштабируемость, надёжность, безопасность.
- **Снижать стоимость изменений**: хорошая архитектура локализует изменения, плохая — заставляет переписывать всё.
- **Обеспечивать коммуникацию**: архитектура — это язык, на котором команда обсуждает систему.

### 2.3. Ключевые аспекты архитектуры

- **Компоненты**: функциональные модули, сервисы, слои.
- **Связи**: как компоненты взаимодействуют (синхронно/асинхронно, через API, сообщения, общую память).
- **Ограничения**: технологический стек, бюджет, сроки, регуляторные требования.
- **Нефункциональные требования**: производительность, доступность, безопасность, сопровождаемость.

### 2.4. Процесс проектирования архитектуры (упрощённо)

1.  **Понять требования** (функциональные и нефункциональные).
2.  **Определить ключевые сценарии** использования.
3.  **Выбрать архитектурный стиль** (монолит, слоистая, микросервисы и т.д.).
4.  **Декомпозировать систему** на компоненты.
5.  **Определить интерфейсы** и протоколы взаимодействия.
6.  **Задокументировать** архитектурные решения (ADR — Architecture Decision Records).
7.  **Валидировать** архитектуру через прототипы и ревью.

---

## 📚 Часть 3. Обзор архитектурных стилей

### 3.1. Монолитная архитектура

**Что это**: Приложение, в котором весь код (UI, бизнес-логика, доступ к данным) собран в единый исполняемый артефакт и развёртывается как одно целое.

**История**: Монолит — исторически первый архитектурный стиль. В 1990-х годах он стал доминирующим подходом для разработки приложений. До появления распределённых систем всё приложение работало на одной машине (мейнфрейме).

**Проблемы, которые решал**: Простота разработки и развёртывания для небольших команд и проектов с чёткими требованиями. Не требовал сложной инфраструктуры.

**Преимущества**:
- Простота разработки и развёртывания.
- Низкий порог входа для новых разработчиков.
- Простая отладка и тестирование (всё в одном процессе).
- Высокая производительность за счёт вызовов функций в памяти.

**Недостатки**:
- Сложность масштабирования: можно масштабировать только всё приложение целиком.
- Высокая связанность: изменение в одном модуле может сломать другие.
- Долгий цикл выпуска: любое изменение требует пересборки и переразвёртывания всего приложения.
- Ограничения по технологическому стеку: всё приложение пишется на одном языке/фреймворке.

**Когда выбирать**: Небольшие и средние проекты, MVP, стартапы на ранней стадии, команды до 10–15 человек.

#### ASCII-схема монолита

```
    ┌─────────────────────────────────────────────────┐
    │            📦 MONOLITH (один артефакт)          │
    │                                                 │
    │  ┌──────────────┐  ┌──────────────┐             │
    │  │   UI / API   │  │  Business    │             │
    │  │   Layer      │──│  Logic       │             │
    │  └──────────────┘  └──────┬───────┘             │
    │                            │                    │
    │                     ┌──────▼───────┐            │
    │                     │  Data Access │            │
    │                     └──────┬───────┘            │
    └────────────────────────────┼────────────────────┘
                                 │
                                 │  SQL
                                 ▼
                          ┌──────────────┐
                          │  🗄️  СУБД    │
                          └──────────────┘

    ✅ Один процесс, один деплой, всё в памяти.
    ⚠️ Сбой модуля роняет всё приложение.
```

### 3.2. N-уровневая (слоистая) архитектура

**Что это**: Приложение разделено на логические слои (уровни), каждый из которых отвечает за свою часть функциональности. Классический пример — трёхуровневая архитектура: Presentation (UI), Business Logic, Data Access.

**История**: В начале 1990-х появилась клиент-серверная архитектура (2-tier). В 2002 году Мартин Фаулер в книге «Patterns of Enterprise Application Architecture» описал N-уровневую архитектуру как эволюцию клиент-серверной модели. В 2000-х трёхуровневая архитектура стала стандартом для веб-приложений.

**Проблемы, которые решал**: Разделение ответственности между UI, бизнес-логикой и данными. Уменьшение связанности между клиентом и бизнес-логикой.

**Преимущества**:
- Чёткое разделение ответственности.
- Возможность независимой разработки и тестирования слоёв.
- Простота замены слоя (например, смена СУБД затрагивает только DAL).
- Хорошая переносимость между средами.

**Недостатки**:
- Дополнительная задержка при передаче данных между слоями.
- Риск «протекания» логики через слои (например, бизнес-логика в UI).
- Сложность изменений, затрагивающих несколько слоёв.
- Чем больше слоёв, тем выше сложность и накладные расходы.

**Когда выбирать**: Корпоративные приложения, системы с чёткими требованиями и стабильной предметной областью. Хорошо подходит для миграции существующих слоистых приложений в облако.

#### Иллюстрация 1. Двухуровневая (2-tier) архитектура

```
    ┌─────────────────────────────────────┐
    │         🖥️  CLIENT TIER             │
    │                                     │
    │   ┌─────────────┐ ┌─────────────┐   │
    │   │     UI      │ │  Бизнес-    │   │
    │   │  (формы,    │ │   логика    │   │
    │   │   кнопки)   │ │  (правила)  │   │
    │   └─────────────┘ └─────────────┘   │
    │                                     │
    └──────────────┬──────────────────────┘
                   │
                   │  SQL-запросы напрямую
                   │  (ODBC / JDBC)
                   │
    ┌──────────────▼──────────────────────┐
    │         🗄️  DATA TIER               │
    │                                     │
    │   ┌─────────────────────────────┐   │
    │   │          СУБД               │   │
    │   │      (PostgreSQL,           │   │
    │   │       MySQL, Oracle)        │   │
    │   └─────────────────────────────┘   │
    │                                     │
    └─────────────────────────────────────┘

    ⚠️ Логика размазана по клиентам — при обновлении нужно
       переустанавливать приложение на каждом компьютере.
```

**Классический пример**: десктопное приложение (старый 1С, Access-приложение), которое подключается к серверу БД по ODBC/JDBC.

#### Иллюстрация 2. Трёхуровневая (3-tier) архитектура

```
    ┌─────────────────────────────────────┐
    │      🖥️  PRESENTATION TIER          │
    │                                     │
    │   ┌─────────────────────────────┐   │
    │   │  Browser / Mobile App       │   │
    │   │  (только отображение        │   │
    │   │   и ввод данных)            │   │
    │   └─────────────────────────────┘   │
    │                                     │
    └──────────────┬──────────────────────┘
                   │
                   │  HTTP / REST / gRPC
                   │  (JSON, Protobuf)
                   │
    ┌──────────────▼──────────────────────┐
    │      ⚙️  BUSINESS LOGIC TIER        │
    │                                     │
    │   ┌─────────────────────────────┐   │
    │   │   Application Server        │   │
    │   │   • правила предметной обл. │   │
    │   │   • расчёты                 │   │
    │   │   • валидация               │   │
    │   │   • авторизация             │   │
    │   └─────────────────────────────┘   │
    │                                     │
    └──────────────┬──────────────────────┘
                   │
                   │  SQL / ORM
                   │  (только сервер!)
                   │
    ┌──────────────▼──────────────────────┐
    │          🗄️  DATA TIER              │
    │                                     │
    │   ┌─────────────────────────────┐   │
    │   │          СУБД               │   │
    │   │   (PostgreSQL, MySQL)       │   │
    │   └─────────────────────────────┘   │
    │                                     │
    └─────────────────────────────────────┘

    ✅ Клиент не знает о БД.
    ✅ Логику можно менять без обновления клиента.
    ✅ Один сервер обслуживает много клиентов.
```

**Классический пример**: любое современное веб-приложение (React SPA + Node.js/Python backend + PostgreSQL).

#### Иллюстрация 3. Сравнение 2-tier и 3-tier

```
    2-tier                          3-tier
    ────────                        ────────

    ┌─────────────┐                 ┌─────────────┐
    │  🖥️ Client  │                 │  🖥️ Client  │
    │ UI + логика │                 │  только UI  │
    └──────┬──────┘                 └──────┬──────┘
           │                               │
           │ напрямую                      │ HTTP
           │                               ▼
           │                        ┌─────────────┐
           │                        │  ⚙️ Server  │
           │                        │  (логика)   │
           │                        └──────┬──────┘
           │                               │
           │                               │ SQL
           ▼                               ▼
    ┌─────────────┐                 ┌─────────────┐
    │  🗄️  СУБД    │                 │  🗄️  СУБД    │
    └─────────────┘                 └─────────────┘

    ⚠️ Связанность:                 ✅ Клиент не знает о БД.
       клиент знает о БД.           ✅ Логика централизована.
```

### 3.3. Микросервисная архитектура

**Что это**: Приложение состоит из набора небольших, независимо развёртываемых сервисов, каждый из которых отвечает за отдельную бизнес-возможность. Сервисы взаимодействуют через лёгкие механизмы (HTTP/REST, gRPC, очереди сообщений).

**История**: Термин «микросервисы» впервые обсуждался на семинаре по архитектуре ПО в мае 2011 года. В 2014 году Джеймс Льюис и Мартин Фаулер опубликовали статью, формализовавшую это понятие. Пионерами подхода стали Netflix и Amazon, которые начали переход от монолитов к микросервисам ещё в 2009 году.

**Проблемы, которые решал**: Ограничения монолитов — сложность масштабирования, долгие циклы выпуска, технологическая привязка, низкая автономность команд.

**Преимущества**:
- Независимое развёртывание и масштабирование каждого сервиса.
- Технологическая гетерогенность: каждый сервис может использовать свой стек.
- Изоляция отказов: сбой одного сервиса не обязательно «роняет» всю систему.
- Автономность команд: небольшие команды могут работать независимо.

**Недостатки**:
- Высокая операционная сложность: требуется оркестрация (Kubernetes), мониторинг, distributed tracing.
- Сложность отладки и тестирования распределённых систем.
- Сетевые задержки и ненадёжность коммуникаций.
- Сложность обеспечения согласованности данных (распределённые транзакции).

**Когда выбирать**: Крупные системы с высокой нагрузкой, много команд, потребность в частых релизах и независимом масштабировании.

#### ASCII-схема микросервисов

```
                       ┌──────────────────────┐
                       │   🖥️  API Gateway    │
                       │   (маршрутизация,    │
                       │    аутентификация)   │
                       └──────┬───────────────┘
                              │
        ┌─────────┬───────────┼───────────┬─────────┐
        │         │           │           │         │
        ▼         ▼           ▼           ▼         ▼
    ┌───────┐ ┌───────┐   ┌───────┐   ┌───────┐ ┌───────┐
    │ 🧩    │ │ 🧩    │   │ 🧩    │   │ 🧩    │ │ 🧩    │
    │Orders │ │Users  │   │Cart   │   │Pay    │ │Notify │
    │Service│ │Service│   │Service│   │Service│ │Service│
    └───┬───┘ └───┬───┘   └───┬───┘   └───┬───┘ └───┬───┘
        │         │           │           │         │
        ▼         ▼           ▼           ▼         ▼
    ┌───────┐ ┌───────┐   ┌───────┐   ┌───────┐ ┌───────┐
    │ 🗄️ DB │ │ 🗄️ DB │   │ 🗄️ DB │   │ 🗄️ DB │ │ 🗄️ DB │
    │orders │ │users  │   │cart   │   │pay    │ │notif  │
    └───────┘ └───────┘   └───────┘   └───────┘ └───────┘

    ✅ Каждый сервис — своя БД, свой стек, свой деплой.
    ✅ Сбой одного сервиса не роняет остальные (при правильной изоляции).
    ⚠️ Требуется: Kubernetes, service mesh, distributed tracing.
```

**Классический пример**: Netflix (700+ микросервисов), Amazon, Uber, Spotify.

### 3.4. Сервис-ориентированная архитектура (SOA)

**Что это**: Архитектурный стиль, в котором приложение представляет собой набор **слабосвязанных, взаимодействующих сервисов**, каждый из которых предоставляет бизнес-возможность через **стандартизованный интерфейс** (обычно через ESB — Enterprise Service Bus). Сервисы могут принадлежать разным системам и даже разным организациям.

**История**: SOA возникла в конце 1990-х — начале 2000-х как ответ на кризис «монолитных ERP-систем». Компании накопили десятки разнородных приложений (CRM, ERP, биллинг, склад), которые не могли обмениваться данными. В 1996 году Gartner впервые ввела термин «service-oriented». Пик популярности — 2005–2012 гг., когда крупные корпорации (банки, телеком, госсектор) строили SOA-ландшафты на базе ESB (IBM WebSphere, Oracle Service Bus, TIBCO).

**Проблемы, которые решала**:
- **Интеграционный хаос**: каждая пара систем общалась по-своему (point-to-point интеграции). SOA предложила единую шину (ESB).
- **Дублирование бизнес-логики**: одна и та же логика (например, расчёт налога) дублировалась в 5 системах. SOA вынесла её в отдельный сервис.
- **Технологическая гетерогенность**: старые системы на мейнфреймах и новые на Java должны были работать вместе. SOA предложила стандарты (SOAP, WSDL, UDDI).
- **Долгие изменения**: изменение бизнес-правила требовало правки всех систем.

**Преимущества**:
- **Повторное использование сервисов**: один сервис «Клиент» используется и биллингом, и CRM, и поддержкой.
- **Слабая связанность**: сервисы общаются через стандартные контракты, а не через прямые вызовы.
- **Интеграция разнородных систем**: ESB умеет преобразовывать протоколы (SOAP ↔ REST ↔ FTP ↔ JMS).
- **Централизованное управление**: мониторинг, безопасность и маршрутизация — на уровне шины.
- **Постепенная модернизация**: можно заменять legacy-системы по одной, не ломая остальные.

**Недостатки**:
- **ESB как единая точка отказа**: если шина падает — вся система стоит.
- **Дороговизна**: лицензии ESB, обучение, инфраструктура — миллионы долларов.
- **Высокая сложность**: SOAP, WSDL, WS-Security, UDDI — тяжёлый стек.
- **«Распределённый монолит»**: если сервисы связаны через ESB слишком плотно, теряются все преимущества.
- **Узкое место по производительности**: ESB добавляет задержку на каждый вызов.
- **Vendor lock-in**: переход с одного ESB на другой — многолетний проект.

**Когда выбирать**:
- Крупные корпорации с десятками разнородных систем.
- Интеграция legacy-систем с современными приложениями.
- Регулируемые отрасли (банки, телеком, госсектор), где важна централизованная безопасность и аудит.
- Когда есть бюджет и команда для поддержки ESB.

**Чем отличается от микросервисов**:

| Критерий | SOA | Микросервисы |
|----------|-----|--------------|
| Размер сервиса | Крупные, бизнес-ориентированные | Мелкие, по одной возможности |
| Коммуникация | Через ESB (SOAP, WSDL) | Напрямую (REST, gRPC, очереди) |
| Управление | Централизованное (шина) | Децентрализованное (хореография) |
| Данные | Общая БД часто допустима | Каждый сервис — своя БД |
| Стек | Унифицированный | Гетерогенный |
| Организация | По слоям | По командам (Conway's Law) |

#### ASCII-схема SOA

```
    ┌─────────────────────────────────────────────────────────────┐
    │                    🖥️  ПОТРЕБИТЕЛИ                          │
    │   ┌──────────┐    ┌──────────┐    ┌──────────────┐         │
    │   │ Веб-прил.│    │ Мобильное│    │ Партнёрская  │         │
    │   │          │    │ прилож.  │    │   система    │         │
    │   └────┬─────┘    └────┬─────┘    └──────┬───────┘         │
    └────────┼───────────────┼─────────────────┼────────────────┘
             │               │                 │
             └───────────────┼─────────────────┘
                             │
                             ▼
    ┌─────────────────────────────────────────────────────────────┐
    │            🚌 ENTERPRISE SERVICE BUS (ESB)                  │
    │                                                             │
    │  ┌────────────┐ ┌────────────┐ ┌────────────┐ ┌──────────┐  │
    │  │Маршрутизация│ │Трансформация│ │Безопасность│ │Мониторинг│  │
    │  │            │ │ протоколов  │ │            │ │          │  │
    │  └────────────┘ └────────────┘ └────────────┘ └──────────┘  │
    │                                                             │
    └──┬────────────┬─────────────┬──────────────┬───────────────┘
       │            │             │              │
       ▼            ▼             ▼              ▼
    ┌──────┐    ┌──────┐      ┌──────┐       ┌──────────┐
    │ ⚙️   │    │ ⚙️   │      │ ⚙️   │       │ Legacy   │
    │Клиен-│    │Заказы│      │Бил-  │       │ (мейн-   │
    │ты    │    │      │      │линг  │       │  фрейм)  │
    │(SOAP)│    │(SOAP)│      │(REST)│       │          │
    └──┬───┘    └──┬───┘      └──┬───┘       └────┬─────┘
       │           │             │                │
       ▼           ▼             ▼                ▼
    ┌──────┐    ┌──────┐      ┌──────┐       ┌──────────┐
    │🗄️ DB │    │🗄️ DB │      │🗄️ DB │       │  Legacy  │
    │клиент│    │заказы│      │биллинг│      │    БД    │
    └──────┘    └──────┘      └──────┘       └──────────┘

    ✅ Единая точка входа и стандарты (SOAP, WSDL, UDDI).
    ⚠️ ESB — единая точка отказа и узкое место по производительности.
```

**Классический пример**: интеграционный ландшафт крупного банка — CRM от одного вендора, АБС от другого, биллинг от третьего, всё связано через IBM WebSphere.

### 3.5. Событийно-ориентированная архитектура (EDA)

**Что это**: Архитектурный стиль, в котором компоненты взаимодействуют через **производство и потребление событий** (events). Производитель (publisher) публикует событие «что-то произошло», а потребители (subscribers) реагируют на него. Производитель **не знает**, кто и как отреагирует.

**История**: Концепция восходит к **паттерну Observer** (1994, «Design Patterns» GoF) и **message-oriented middleware** (MOM) 1990-х (IBM MQSeries, TIBCO Rendezvous). Термин «event-driven architecture» популяризирован Gartner в середине 2000-х. Новый виток популярности — с появлением **Apache Kafka** (2011), **event sourcing** и **CQRS** (Грег Янг, 2010), а также реактивных манифестов (2014).

**Проблемы, которые решала**:
- **Жёсткая связанность**: в монолите и слоистой архитектуре вызывающий код должен знать, кого вызывать. EDA убирает это знание.
- **Синхронные блокировки**: если сервис А ждёт ответа от Б, а Б тормозит — тормозит всё. EDA делает взаимодействие асинхронным.
- **Масштабирование пиков**: всплеск заказов «роняет» синхронную систему. Очередь событий сглаживает нагрузку.
- **Добавление новых потребителей**: чтобы новый сервис получал данные, приходилось править источник. EDA позволяет подключаться «сбоку».
- **Аудит и восстановление состояния**: классические БД хранят только текущее состояние. Event Sourcing хранит всю историю изменений.

**Преимущества**:
- **Слабая связанность**: publisher и subscriber не знают друг о друге.
- **Асинхронность**: publisher не ждёт ответа — система отзывчивее.
- **Масштабируемость**: легко добавлять новых потребителей и горизонтально масштабировать обработчиков.
- **Отказоустойчивость**: если потребитель упал — события накопятся в очереди и будут обработаны позже.
- **Реактивность в реальном времени**: подходит для streaming, IoT, аналитики.
- **Естественный аудит**: поток событий = журнал всех изменений.
- **Event Sourcing + CQRS**: позволяют строить системы с полной историей и раздельными моделями чтения/записи.

**Недостатки**:
- **Сложность отладки**: нет линейного стека вызовов, ошибка может «всплыть» через 3 сервиса и 5 минут.
- **Согласованность eventual**: данные в разных сервисах временно расходятся. Это нужно принять как норму.
- **Дублирование событий**: at-least-once delivery требует **идемпотентности** обработчиков.
- **Порядок событий**: гарантия порядка — сложная задача (Kafka даёт порядок только внутри партиции).
- **Схема событий**: изменение формата события ломает потребителей. Нужны schema registry и версионирование.
- **Операционная сложность**: Kafka/RabbitMQ/Pulsar — отдельная инфраструктура, требующая экспертизы.
- **«Event Hell»**: если событий слишком много и они не документированы — разобраться невозможно.

**Ключевые паттерны EDA**:
- **Pub/Sub**: издатель публикует событие в топик, все подписчики получают копию.
- **Event Streaming**: события хранятся в логе (Kafka), можно перечитать с любого места.
- **Event Sourcing**: состояние системы = последовательность событий, а не текущий снимок.
- **CQRS**: разделение модели чтения и модели записи; запись порождает события, чтение строит проекции.
- **Saga**: распределённая транзакция через последовательность событий с компенсациями.
- **Outbox Pattern**: гарантия атомарности «запись в БД + публикация события».

**Когда выбирать**:
- Системы с высокой нагрузкой и пиками (e-commerce, биллинг, телеком).
- Real-time аналитика, streaming-обработка, IoT.
- Интеграция множества сервисов, где важна слабая связанность.
- Системы, где нужен полный аудит изменений (финансы, медицина, логистика).
- Когда есть команда, готовая поддерживать Kafka/Pulsar.

**Чем отличается от SOA и микросервисов**:

| Критерий | SOA | Микросервисы | EDA |
|----------|-----|--------------|-----|
| Стиль взаимодействия | Синхронный (запрос-ответ) | Синхронный + асинхронный | Преимущественно асинхронный |
| Связанность | Через ESB | Через API | Через события |
| Знание о потребителе | Знает ESB | Знает endpoint | Не знает |
| Централизация | Высокая (шина) | Низкая | Низкая |
| Основная единица | Сервис | Сервис | Событие |
| Согласованность | Строгая | Строгая / eventual | Eventual |

#### ASCII-схема EDA

```
    ┌──────────────────────────────────────────────────────────────┐
    │              📤 ПРОИЗВОДИТЕЛИ СОБЫТИЙ                        │
    │   ┌────────────┐  ┌────────────┐  ┌──────────────┐          │
    │   │  Сервис    │  │  Сервис    │  │  IoT-        │          │
    │   │  'Заказы'  │  │  'Платежи' │  │  устройства  │          │
    │   └─────┬──────┘  └─────┬──────┘  └──────┬───────┘          │
    └─────────┼───────────────┼─────────────────┼─────────────────┘
              │               │                 │
              ▼               ▼                 ▼
    ┌──────────────────────────────────────────────────────────────┐
    │      📨 EVENT BROKER / LOG (Kafka, RabbitMQ, Pulsar)         │
    │                                                              │
    │   ┌────────────────┐  ┌────────────────┐  ┌───────────────┐  │
    │   │Topic:          │  │Topic:          │  │Topic:         │  │
    │   │order.created   │  │payment.complete│  │sensor.data    │  │
    │   └────┬──────┬────┘  └────┬──────┬────┘  └───┬───────┬───┘  │
    └────────┼──────┼────────────┼──────┼───────────┼───────┼──────┘
             │      │            │      │           │       │
             │      │            │      │           │       │
             ▼      │            ▼      │           ▼       │
    ┌────────────┐  │    ┌────────────┐ │   ┌────────────┐  │
    │Сервис      │  │    │Сервис      │ │   │Сервис      │  │
    │'Уведомле-  │  │    │'Аналитика' │ │   │'Склад'     │  │
    │  ния'      │  │    │            │ │   │            │  │
    └────────────┘  │    └────────────┘ │   └────────────┘  │
                    │                   │                   │
                    ▼                   ▼                   ▼
              ┌────────────┐    ┌────────────┐      ┌────────────┐
              │Data Lake / │    │ Дашборды   │      │  Архив /   │
              │  Хранилище │    │ real-time  │      │  Аудит     │
              └────────────┘    └────────────┘      └────────────┘

    ✅ Производители не знают о потребителях.
    ✅ Один топик может читаться несколькими сервисами независимо.
    ⚠️ Согласованность — eventual. Нужны идемпотентные обработчики.
```

**Классический пример**:
- **Uber**: каждое действие водителя/пассажира = событие в Kafka, десятки сервисов реагируют (ценообразование, ETA, уведомления, аналитика).
- **Netflix**: миллиарды событий просмотра → рекомендации, биллинг, мониторинг.
- **Банк**: транзакция → событие `payment.completed` → антифрод, уведомление, обновление баланса, отчётность.

---

### 3.6. Соотношение нефункциональных требований и выбора архитектуры

**Главная мысль**: Архитектуру определяют **не функциональные требования** («что система делает»), а **нефункциональные** («как она это делает»). Два приложения с одинаковым набором функций (например, «интернет-магазин») будут иметь **разную архитектуру**, если у одного 100 пользователей, а у другого — 10 миллионов.

> «Функциональность можно добавить в любой архитектуре. Нефункциональные требования — нельзя.» — Ральф Джонсон

#### 3.6.1. Почему NFR определяют архитектуру?

Функциональные требования (FR) можно реализовать в любом стиле — монолит тоже умеет считать корзину. А вот **NFR накладывают ограничения**, которые «отсекают» целые классы архитектурных решений:

| NFR | Что это значит для архитектуры |
|-----|-------------------------------|
| «100 000 запросов/сек» | Нужна горизонтальная масштабируемость → монолит отпадает |
| «Доступность 99.99%» | Нужна изоляция отказов → единый процесс опасен |
| «Время отклика < 100 мс на 3 континентах» | Нужны edge-компоненты, CDN, репликация |
| «Аудит всех изменений за 5 лет» | Нужен event sourcing |
| «Соответствие GDPR, данные только в ЕС» | Нужна географическая изоляция данных |
| «Команда из 200 разработчиков» | Нужна независимая разработка → Conway's Law → микросервисы |
| «Бюджет: 3 разработчика и 2 месяца» | Всё, кроме монолита, отпадает |

#### 3.6.2. Классификация NFR по ISO/IEC 25010

Стандарт **ISO/IEC 25010** («Systems and software Quality Requirements and Evaluation», 2011, обновлён в 2023) определяет 8 характеристик качества:

| Характеристика | Подхарактеристики | Влияние на архитектуру |
|----------------|-------------------|------------------------|
| **Performance Efficiency** | Time behaviour, Resource utilization, Capacity | Кэширование, шардирование, асинхронность |
| **Reliability** | Maturity, Availability, Fault tolerance, Recoverability | Репликация, circuit breaker, изоляция |
| **Security** | Confidentiality, Integrity, Non-repudiation, Accountability | Аутентификация, шифрование, аудит |
| **Usability** | Learnability, Operability, Accessibility | Тонкий клиент, SSR, PWA |
| **Maintainability** | Modularity, Reusability, Analysability, Modifiability, Testability | Слои, модули, микросервисы |
| **Portability** | Adaptability, Installability, Replaceability | Контейнеризация, абстракции |
| **Compatibility** | Co-existence, Interoperability | Стандартные протоколы, ESB |
| **Functional Suitability** | Completeness, Correctness, Appropriateness | (ближе к FR) |

> **Ключевое**: NFR **конфликтуют** друг с другом. Улучшение одного почти всегда ухудшает другое. Это и есть **trade-offs** — суть архитектурной работы.

#### 3.6.3. Главные trade-offs в архитектуре

| Улучшаем | Ухудшаем | Пример |
|----------|----------|--------|
| Производительность | Сопровождаемость | Оптимизации «в лоб» вместо чистого кода |
| Масштабируемость | Простота | Микросервисы вместо монолита |
| Доступность | Согласованность | CAP-теорема: AP вместо CP |
| Безопасность | Удобство | MFA вместо «входа в один клик» |
| Гибкость (изменяемость) | Производительность | Абстракции вместо прямых вызовов |
| Скорость разработки | Качество | MVP с техдолгом |

#### 3.6.4. Quality Attribute Scenarios (QAS)

NFR нужно формулировать **измеримо**, иначе их нельзя ни спроектировать, ни проверить. Формат **Quality Attribute Scenario** (из книги «Software Architecture in Practice», Bass, Clements, Kazman):

```
Источник → Стимул → Артефакт → Окружение → Реакция → Мера реакции
```

**Пример плохого NFR**: «Система должна быть быстрой».
**Пример хорошего NFR через QAS**:

| Элемент | Значение |
|---------|----------|
| Источник | Пользователь |
| Стимул | Отправляет запрос на поиск |
| Артефакт | Сервис поиска |
| Окружение | Нормальная нагрузка (10 000 RPS) |
| Реакция | Возвращает результаты |
| Мера реакции | < 200 мс в 95% случаев (p95) |

**Ещё пример (доступность)**:

| Элемент | Значение |
|---------|----------|
| Источник | Внешний сбой |
| Стимул | Отказ одного из дата-центров |
| Артефакт | Платёжный сервис |
| Окружение | Пиковая нагрузка |
| Реакция | Трафик переключается на резервный ДЦ |
| Мера реакции | Downtime < 30 сек, потеря данных = 0 |

#### 3.6.5. Как NFR влияют на выбор стиля: матрица

| NFR (приоритет) | Монолит | N-уровневая | SOA | Микросервисы | EDA |
|-----------------|:-------:|:-----------:|:---:|:------------:|:---:|
| **Высокая производительность (низкая задержка)** | ★★★★ | ★★★ | ★★ | ★★★ | ★★★★ |
| **Горизонтальная масштабируемость** | ★★ | ★★★ | ★★★★ | ★★★★★ | ★★★★★ |
| **Высокая доступность (99.99%)** | ★★ | ★★★ | ★★★ | ★★★★★ | ★★★★★ |
| **Строгая согласованность данных** | ★★★★★ | ★★★★★ | ★★★★ | ★★ | ★ |
| **Частые релизы (несколько раз в день)** | ★ | ★★ | ★★ | ★★★★★ | ★★★★★ |
| **Простота поддержки (маленькая команда)** | ★★★★★ | ★★★★ | ★★ | ★★ | ★★ |
| **Полный аудит изменений** | ★★ | ★★ | ★★★ | ★★★ | ★★★★★ |
| **Гибкость в выборе стека** | ★ | ★★ | ★★★ | ★★★★★ | ★★★★★ |
| **Безопасность (изоляция, аудит, шифрование)** | ★★★ | ★★★ | ★★★★★ | ★★★★ | ★★★★ |
| **Низкая стоимость инфраструктуры** | ★★★★★ | ★★★★ | ★★ | ★★ | ★★★ |

> **Правило**: Сначала определите **2–3 доминирующих NFR** (те, ради которых проект вообще существует). Затем смотрите на строки этой таблицы — стиль с наибольшим числом звёзд в этих строках и будет кандидатом. Остальные NFR — вторичны, ими можно пожертвовать.

#### 3.6.6. Алгоритм выбора архитектуры по NFR

**Шаг 1. Соберите NFR** и переведите их в **QAS** (измеримые сценарии).

**Шаг 2. Приоритизируйте NFR** по MoSCoW или методом Кано. Выделите 2–3 **архитектурно значимых** (Architecturally Significant Requirements, ASR).

**Шаг 3. Определите ключевые trade-offs**. Например:
- Высокая доступность ↔ строгая согласованность (CAP).
- Низкая задержка ↔ гибкость (абстракции стоят времени).

**Шаг 4. Сузьте круг стилей** по матрице из п. 3.6.5.

**Шаг 5. Проверьте ограничения** (Constraints):
- Бюджет, сроки, размер команды.
- Технологический стек компании.
- Регуляторные требования (GDPR, PCI DSS, ФЗ-152).
- Квалификация команды.

**Шаг 6. Проведите ATAM-сессию** (Architecture Tradeoff Analysis Method, SEI) — быстрая валидация архитектуры на соответствие NFR через сценарии. Минимум: соберите 5–10 QAS и проверьте, что выбранный стиль их закрывает.

**Шаг 7. Задокументируйте решение** через ADR с явным указанием, **какими NFR пришлось пожертвовать** и почему.

#### 3.6.7. Практический пример: интернет-магазин

**Контекст**:
- MVP, 5 разработчиков, срок — 4 месяца.
- Ожидаемая нагрузка: 1000 пользователей в день.
- Бюджет: минимальный.
- NFR-приоритеты: **скорость разработки**, **простота поддержки**.

**Решение**: **Монолит** (например, Django + PostgreSQL). NFR «масштабируемость» и «доступность 99.99%» **осознанно отложены** — их не требуют на текущем этапе.

**Через 2 года**:
- 5 млн пользователей, 50 разработчиков.
- NFR-приоритеты: **масштабируемость**, **частые релизы**, **доступность 99.99%**.

**Решение**: **Эволюция в микросервисы + EDA**. Монолит разбивается по bounded contexts (Orders, Catalog, Payments, Users), добавляется Kafka для асинхронных сценариев (уведомления, аналитика).

**Вывод**: архитектура **меняется вместе с NFR**. Один и тот же продукт на разных этапах жизни имеет разные архитектуры. Не существует «правильной» архитектуры «на все времена».

#### 3.6.8. Типичные ошибки

| Ошибка | Что происходит | Как избежать |
|--------|----------------|--------------|
| Выбор стиля «потому что модно» | Микросервисы для 3 пользователей | Начинайте с NFR, а не с технологий |
| Игнорирование NFR на старте | Переписывание через год | Соберите NFR вместе с FR на этапе анализа |
| Немеримые NFR | «Быстро», «надёжно», «удобно» | Переводите в QAS: «p95 < 200 мс» |
| Копирование архитектуры гигантов | «Сделаем как Netflix» без их задач | Контекст важнее best practices |
| Преждевременная оптимизация | Сложная архитектура «на будущее» | YAGNI для архитектуры: стройте под текущие NFR |
| Игнорирование ограничений | Красивая архитектура, которую некому поддерживать | Учитывайте команду и бюджет |

#### 3.6.9. Мини-чек-лист архитектора

Перед выбором стиля ответьте на вопросы:

1. Какие **2–3 NFR** критичны для бизнеса? (не «все важны» — выберите!)
2. Как они сформулированы **измеримо** (QAS)?
3. Какие **trade-offs** мы принимаем?
4. Какой **бюджет, сроки и команда**?
5. Какие **регуляторные ограничения**?
6. Как архитектура будет **эволюционировать** через 1, 3, 5 лет?
7. Какие NFR мы **осознанно отложили** и на каких условиях вернёмся к ним?

---

## 📊 Часть 4. Сравнение архитектурных стилей

| Критерий | Монолит | N-уровневая | SOA | Микросервисы | EDA |
|----------|---------|-------------|-----|--------------|-----|
| **Простота разработки** | ★★★★★ | ★★★★ | ★★ | ★★ | ★★ |
| **Простота развёртывания** | ★★★★★ | ★★★★ | ★ | ★★ | ★★ |
| **Масштабируемость** | ★★ | ★★★ | ★★★★ | ★★★★★ | ★★★★★ |
| **Отказоустойчивость** | ★★ | ★★★ | ★★★ | ★★★★★ | ★★★★★ |
| **Независимость развёртывания** | ★ | ★★ | ★★★ | ★★★★★ | ★★★★★ |
| **Технологическая гибкость** | ★ | ★★ | ★★★ | ★★★★★ | ★★★★★ |
| **Слабая связанность** | ★ | ★★ | ★★★ | ★★★★ | ★★★★★ |
| **Сложность эксплуатации** | ★★★★★ (min) | ★★★★ | ★ | ★ | ★ |
| **Стоимость начального этапа** | ★★★★★ | ★★★★ | ★★ | ★★ | ★★ |
| **Автономность команд** | ★★ | ★★★ | ★★★ | ★★★★★ | ★★★★★ |
| **Аудит и история** | ★★ | ★★ | ★★★ | ★★★ | ★★★★★ |
| **Согласованность данных** | ★★★★★ | ★★★★★ | ★★★★ | ★★★ | ★★ |
| **Порог входа для разработчика** | ★★★★★ | ★★★★ | ★★ | ★★★ | ★★ |
| **Зрелость практик** | ★★★★★ | ★★★★★ | ★★★★ | ★★★★ | ★★★ |

> **Вывод**:
> - **Монолит** — старт любого проекта. Дёшево, быстро, понятно.
> - **N-уровневая** — когда монолит нужно структурировать.
> - **SOA** — интеграция legacy в крупной организации (дорого, но надёжно).
> - **Микросервисы** — когда команд много, а релизы нужны часто.
> - **EDA** — когда нужна real-time реакция, слабая связанность и полный аудит.
>
> Стили **не взаимоисключающие**: современная система может быть микросервисной и при этом событийно-ориентированной.

---

## 🧪 Часть 5. Практические задания

### Задание 1. Рефакторинг с SOLID (20 мин)

Дан класс:
```python
class Order:
    def __init__(self, items):
        self.items = items

    def calculate_total(self):
        return sum(item.price for item in self.items)

    def save_to_db(self):
        # сохранение в базу данных
        pass

    def print_invoice(self):
        # печать счёта
        pass
```
**Задача**: Определите, какие принципы SOLID нарушены, и отрефакторите код.
## Решение

**Нарушен SRP (Single Responsibility):** класс Order делает три вещи — считает сумму, сохраняет в БД, печатает счёт. Три причины для изменения.

**Нарушен DIP (Dependency Inversion):** Order жёстко знает про базу данных, вместо того чтобы зависеть от абстракции.

**Рефакторинг код внизу.**
### Задание 2. DRY, KISS, YAGNI (15 мин)

Даны три фрагмента кода. Для каждого определите, какой принцип (DRY, KISS, YAGNI) нарушен, и предложите исправление.

**Фрагмент A**:
```python
def create_user(email):
    if not email or "@" not in email:
        raise ValueError("Invalid email")
    # ...

def update_user_email(user, email):
    if not email or "@" not in email:
        raise ValueError("Invalid email")
    # ...
```

**Фрагмент B**:
```python
def get_discount(user):
    return 10 if user.is_premium and user.years > 5 and user.orders > 100 else (5 if user.is_premium and user.orders > 50 else 0)
```

**Фрагмент C**:
```python
class PaymentProcessor:
    def process(self, method):
        if method == "credit_card":
            # обработка карты
            pass
        elif method == "paypal":
            # обработка PayPal
            pass
        elif method == "crypto":
            # обработка крипты (никто не просил, но на будущее)
            pass
```
## Решение

### Фрагмент A — нарушен DRY

Валидация email продублирована в `create_user` и в `update_user_email`. Это одно и то же знание в двух местах. Выношу в отдельную функцию `validate_email`.

### Фрагмент B — нарушен KISS

Вложенный тернарный оператор нечитаем. Переписываю на обычные `if`.

### Фрагмент C — нарушен YAGNI
**коды внизу**
### Задание 3. Выбор архитектурного стиля (25 мин)

Работа в группах. Даны три сценария. Для каждого выберите наиболее подходящий архитектурный стиль и обоснуйте выбор.

**Сценарий A**: Стартап из 5 человек создаёт MVP мобильного приложения для доставки еды. Сроки — 3 месяца, бюджет ограничен.

**Сценарий B**: Крупная страховая компания модернизирует существующую систему управления полисами. Система работает на устаревшем стеке, есть 20 разработчиков, требуется постепенная миграция.

**Сценарий C**: Стриминговый сервис с миллионами пользователей. Требуется независимое масштабирование рекомендаций, транскодинга видео и биллинга. 50+ разработчиков в 10 командах.

## Решение

### Сценарий A. Стартап, доставка еды (5 человек, 3 месяца, ограниченный бюджет)

**Выбор: монолит.**

Обоснование:
- Мало людей — микросервисы не окупаются (нужна отдельная команда под каждый сервис).
- Мало времени — нет возможности поднимать Kubernetes, Kafka, tracing.
- Мало денег — монолит дешевле всего.
- NFR-приоритеты: скорость разработки, простота, низкая стоимость.

### Сценарий B. Страховая, 20 разработчиков, куча legacy

**Выбор: SOA.**

Обоснование:
- Много старых систем на разных языках — нужна интеграционная шина (ESB).
- Требуется постепенная миграция — SOA позволяет заменять системы по одной.
- Регулируемая отрасль — нужна централизованная безопасность и аудит.
- 20 разработчиков — уже больно для монолита, но ещё не тот масштаб для микросервисов.

### Сценарий C. Стриминг, миллионы юзеров, 10 команд

**Выбор: микросервисы + EDA.**

Обоснование:
- 10 команд — каждой нужен свой сервис (Conway's Law).
- Требуется независимое масштабирование рекомендаций, транскодинга, биллинга.
- Миллионы пользователей — нужна горизонтальная масштабируемость.
- Real-time аналитика — нужен event streaming (Kafka).

### Задание 4. Проектирование архитектуры (30 мин)

Выберите учебный проект (или используйте сценарий из задания 3). Выполните следующие шаги:
1.  Определите ключевые функциональные и нефункциональные требования.
2.  Выберите архитектурный стиль.
3.  Нарисуйте диаграмму компонентов (ASCII-схема или текстовое описание).
4.  Определите интерфейсы между компонентами.
5.  Задокументируйте одно ключевое архитектурное решение в формате ADR.

**Шаблон ADR**:
```
# ADR-001: Выбор архитектурного стиля
## Контекст
## Решение
## Статус
## Последствия
```

### Задание 5. Выбор между SOA и микросервисами (20 мин)

**Сценарий**: Крупный банк имеет 40 legacy-систем (АБС, CRM, биллинг, скоринг, антифрод и т.д.), написанных на разных языках. Нужно интегрировать их в единую экосистему для мобильного приложения.

**Вопросы**:
1. Что выберете: SOA или микросервисы? Обоснуйте.
2. Какие риски у каждого варианта?
3. Как бы вы поэтапно мигрировали?
## Решение

### 1. Что выбрать?

**SOA.**

Обоснование:
- 40 legacy-систем на разных языках — нужна интеграционная шина (ESB) с поддержкой SOAP, JMS, FTP, REST.
- Микросервисы = переписать всё с нуля — это годы и миллиарды.
- SOA позволяет обернуть legacy в сервисы через фасады и мигрировать постепенно.
- Банк — регулируемая отрасль, нужна централизованная безопасность и аудит через шину.

### 2. Риски

| Вариант | Риски |
|---------|-------|
| SOA | ESB — единая точка отказа; дорогие лицензии; vendor lock-in; «распределённый монолит»; узкое место по производительности |
| Микросервисы | Переписывание 40 систем нереально; команда не готова к K8s; риск не завершить миграцию; потеря данных при переносе legacy |

### 3. План миграции

1. Инвентаризация всех 40 систем (интерфейсы, владельцы данных).
2. Разбить на домены (клиенты, счета, платежи, кредиты, карты).
3. Развернуть ESB.
4. Обернуть каждую legacy в сервис-фасад (SOAP/REST).
5. Вынести общие сервисы («Клиент», «Счёт», «Справочник валют»).
6. Мобильное приложение → API Gateway → ESB → сервисы.
7. Постепенно заменять legacy на микросервисы за ESB (Strangler Fig).



### Задание 6. Проектирование EDA-системы (25 мин)

**Сценарий**: Сервис такси. Есть 4 сервиса: Заказы, Водители, Ценообразование, Уведомления.

**Задача**:
1. Определите, какие события должны публиковаться (минимум 5).
2. Для каждого события укажите: producer, topic, consumers.
3. Опишите, что произойдёт, если сервис Уведомления временно упадёт.
4. Как обеспечить идемпотентность обработчика?

**Шаблон**:
```
| Событие | Producer | Topic | Consumers | Критичность |
|---------|----------|-------|-----------|-------------|
| ride.requested | Заказы | ride.requests | Водители, Ценообразование | High |
```
## Решение

### 1–2. События

| Событие | Producer | Topic | Consumers | Критичность |
|---------|----------|-------|-----------|-------------|
| ride.requested | Заказы | ride.requests | Водители, Цены | High |
| ride.accepted | Водители | ride.accepted | Заказы, Уведомления | High |
| ride.started | Водители | ride.started | Заказы, Уведомления | Medium |
| ride.completed | Заказы | ride.completed | Биллинг, Уведомления, Аналитика | High |
| price.calculated | Цены | price.updated | Заказы, Аналитика | Medium |
| driver.location.updated | Водители | driver.locations | Цены, Заказы | High |

### 3. Что если сервис Уведомления упадёт?

- События копятся в Kafka (retention 7 дней).
- Другие сервисы работают — они не ждут уведомлений.
- Когда Уведомления поднимется — прочитает с последнего offset и разошлёт всё пропущенное.
- Потери нет, только задержка.

### 4. Идемпотентность

Kafka даёт at-least-once — событие может прийти дважды. Решение — таблица обработанных event_id.

**код внизу**
### Задание 7. NFR → Архитектура (30 мин)

**Цель**: научиться выбирать архитектурный стиль исходя из нефункциональных требований.

**Даны 4 сценария.** Для каждого:
1. Выделите **2–3 доминирующих NFR**.
2. Переформулируйте их в **QAS** (в формате: источник → стимул → артефакт → окружение → реакция → мера).
3. Выберите архитектурный стиль, обоснуйте через матрицу из п. 3.6.5.
4. Укажите **2 trade-off**, на которые вы сознательно идёте.
5. Перечислите NFR, которые вы **отложили** и почему.

**Сценарий A. Медицинская CRM для небольшой клиники**
- 30 врачей, 5000 пациентов.
- Хранение медкарт (персональные данные, ФЗ-152).
- Бюджет: 1 млн руб., срок — 3 месяца, команда — 3 человека.
- Критично: **безопасность**, **простота**, **низкая стоимость**.

**Сценарий B. Сервис такси в городе-миллионнике**
- 100 000 поездок в день, пики в часы rush hour.
- Real-time отслеживание водителей.
- Критично: **масштабируемость**, **низкая задержка**, **доступность 99.9%**.
- Команда: 30 разработчиков в 6 командах.

**Сценарий C. Банковская система расчётов между юрлицами**
- 500 000 транзакций в день, строгий аудит.
- Требования ЦБ РФ, PCI DSS.
- Интеграция с 20 legacy-системами.
- Критично: **согласованность**, **аудит**, **безопасность**.

**Сценарий D. IoT-платформа для умных счётчиков**
- 1 млн устройств, каждое отправляет данные раз в 15 минут.
- Real-time аналитика потребления.
- Критично: **масштабируемость**, **отказоустойчивость**, **event streaming**.

**Шаблон ответа**:
```
| Сценарий | Доминирующие NFR | QAS | Стиль | Trade-offs | Отложенные NFR |
|----------|------------------|-----|-------|------------|----------------|
| A        |                  |     |       |            |                |
```

**Обсуждение**: сравните ответы разных команд — почему для одного и того же сценария могут быть выбраны разные стили? Что определяет выбор в первую очередь — NFR или ограничения?

---

## Решение

### Сценарий A. Медкарты для клиники

**Доминирующие NFR:** безопасность, простота, стоимость.

**QAS (безопасность):**
- Источник: внешний атакующий
- Стимул: попытка открыть чужую медкарту
- Артефакт: API медкарт
- Окружение: обычная работа
- Реакция: доступ заблокирован, запись в лог
- Мера: 100% попыток без токена отклонено, лог за 1 сек

**Стиль:** монолит + N-уровневая. Бюджет 1 млн, 3 человека, 3 месяца — микросервисы нереальны.

**Trade-offs:** простота <-> масштаб; безопасность (MFA) <-> удобство.

**Отложенные NFR:** масштабируемость, 99,99% доступность.

### Сценарий B. Такси в миллионнике

**Доминирующие NFR:** масштабируемость, низкая задержка, доступность 99,9%.

**QAS (задержка):**
- Источник: пассажир
- Стимул: запрос поездки
- Артефакт: сервис Заказов
- Окружение: час пик
- Реакция: подбор водителя, расчёт цены
- Мера: p95 < 500 мс, p99 < 1 сек

**Стиль:** микросервисы + EDA.

**Trade-offs:** масштаб <-> согласованность; низкая задержка <-> гибкость (Redis для геолокации).

**Отложенные NFR:** полный аудит, сложная аналитика.

### Сценарий C. Банк

**Доминирующие NFR:** согласованность, аудит, безопасность.

**QAS (согласованность):**
- Источник: бухгалтер
- Стимул: перевод 1 млн руб.
- Артефакт: платёжный сервис
- Окружение: обычная работа
- Реакция: транзакция зафиксирована атомарно
- Мера: 0 потерянных транзакций, ACID, аудит-лог

**Стиль:** SOA + N-уровневая. НЕ микросервисы — нужна строгая согласованность (★5 vs ★2).

**Trade-offs:** CP вместо AP (жертвуем доступностью); безопасность <-> скорость релизов.

**Отложенные NFR:** масштаб за пределы ДЦ.

### Сценарий D. IoT-счётчики

**Доминирующие NFR:** масштабируемость, отказоустойчивость, event streaming.

**QAS (масштабируемость):**
- Источник: 1 млн счётчиков
- Стимул: отправка показаний раз в 15 мин
- Артефакт: ingestion-сервис
- Окружение: пиковая нагрузка
- Реакция: приём и запись в поток
- Мера: 1,1 млн msg/сек, потеря < 0,001%

**Стиль:** EDA + микросервисы. Kafka — центральный брокер.

**Trade-offs:** масштаб <-> согласованность; отказоустойчивость <-> стоимость (репликация ×3).

**Отложенные NFR:** строгая согласованность, низкая задержка.

### Итоговая таблица

| Сценарий | Доминирующие NFR | Стиль | Trade-offs | Отложенные NFR |
|----------|------------------|-------|------------|----------------|
| A. Медкарты | Безопасность, простота | Монолит | Простота<->масштаб; MFA↔удобство | Масштаб, 99,99% |
| B. Такси | Масштаб, скорость | Микросервисы + EDA | Масштаб <->согласованность | Аудит, аналитика |
| C. Банк | Согласованность, аудит | SOA | CP<->AP; ревью↔скорость | Масштаб за ДЦ |
| D. IoT | Масштаб, отказоустойчивость | EDA | Масштаб↔согласованность; репликация<->стоимость | Строгая согласованность |

### Обсуждение

Разные команды выбирают разные стили для одного сценария, потому что:
1. По-разному расставляют приоритеты NFR.
2. Разные ограничения (деньги, люди, сроки).
3. Разное отношение к риску.
4. Разный горизонт планирования.

**Что важнее — NFR или ограничения?** Идеально — NFR (что система должна уметь). Но ограничения часто «отрезают» правильный стиль. Хороший архитектор собирает NFR, проверяет ограничения и записывает в ADR, чем пожертвовал.

In [9]:
# Задание 6

# Идемпотентный обработчик события
processed_events = set()  # в реальной системе — таблица в БД

def send_notification(user_id, text):
    print(f"[Notification] {user_id}: {text}")

def handle_ride_completed(event):
    event_id = event["id"]
    if event_id in processed_events:
        print(f"[Skip] Событие {event_id} уже обработано")
        return
    processed_events.add(event_id)
    send_notification(event["user_id"], "Поездка завершена")

# Проверка
event = {"id": "evt-001", "user_id": "user-42"}
handle_ride_completed(event)  # отправит уведомление
handle_ride_completed(event)  # пропустит — уже обработано

[Notification] user-42: Поездка завершена
[Skip] Событие evt-001 уже обработано


In [8]:
# Задание 2

# Фрагмент A — исправление DRY
def validate_email(email: str) -> None:
    if not email or "@" not in email:
        raise ValueError("Некорректный email")

def create_user(email: str):
    validate_email(email)
    print("Пользователь создан")

def update_user_email(user, email: str):
    validate_email(email)
    print("Email обновлён")

# Фрагмент B — исправление KISS
class User:
    def __init__(self, is_premium, years, orders):
        self.is_premium = is_premium
        self.years = years
        self.orders = orders

def get_discount(user) -> int:
    if user.is_premium and user.years > 5 and user.orders > 100:
        return 10
    if user.is_premium and user.orders > 50:
        return 5
    return 0

# Фрагмент C — исправление YAGNI
class PaymentProcessor:
    def process(self, method: str) -> None:
        if method == "credit_card":
            print("Обработка карты")
        elif method == "paypal":
            print("Обработка PayPal")
        else:
            raise ValueError(f"Неизвестный метод: {method}")

# Проверка
validate_email("test@example.com")
print("Discount:", get_discount(User(True, 6, 200)))
PaymentProcessor().process("credit_card")

Discount: 10
Обработка карты


In [7]:
# Задание 1


from dataclasses import dataclass
from typing import Protocol

@dataclass
class Item:
    name: str
    price: float

@dataclass
class Order:
    items: list[Item]

    def calculate_total(self) -> float:
        return sum(item.price for item in self.items)

class OrderRepository(Protocol):
    def save(self, order: Order) -> None: ...

class InvoicePrinter(Protocol):
    def print(self, order: Order, total: float) -> None: ...

class PostgresOrderRepository:
    def save(self, order: Order) -> None:
        print(f"[DB] Сохранил заказ на {len(order.items)} товаров")

class PdfInvoicePrinter:
    def print(self, order: Order, total: float) -> None:
        print(f"[PDF] Счёт на сумму {total}")

class OrderService:
    def __init__(self, repo: OrderRepository, printer: InvoicePrinter):
        self.repo = repo
        self.printer = printer

    def checkout(self, order: Order) -> None:
        total = order.calculate_total()
        self.repo.save(order)
        self.printer.print(order, total)

# Проверка
service = OrderService(PostgresOrderRepository(), PdfInvoicePrinter())
service.checkout(Order([Item("Книга", 500), Item("Ручка", 50)]))

[DB] Сохранил заказ на 2 товаров
[PDF] Счёт на сумму 550


## 📖 Часть 6. Литература

### Основная

1.  Martin, R. C. (2000). *Design Principles and Design Patterns*. — [PDF (архив Object Mentor)](https://web.archive.org/web/20150906155800/http://www.objectmentor.com/resources/articles/Principles_and_Patterns.pdf)
2.  Martin, R. C. (2017). *Clean Architecture: A Craftsman's Guide to Software Structure and Design*. Prentice Hall. — [O'Reilly](https://www.oreilly.com/library/view/clean-architecture-a/9780134494272/) | [Amazon](https://www.amazon.com/Clean-Architecture-Craftsmans-Software-Structure/dp/0134494164)
3.  Hunt, A., & Thomas, D. (1999). *The Pragmatic Programmer*. Addison-Wesley. — [Официальная страница (20th Anniversary Edition)](https://pragprog.com/titles/tpp20/the-pragmatic-programmer-20th-anniversary-edition/)
4.  Fowler, M. (2002). *Patterns of Enterprise Application Architecture*. Addison-Wesley. — [Официальная страница книги](https://martinfowler.com/books/eaa.html)
5.  Lewis, J., & Fowler, M. (2014). *Microservices*. — [Статья на martinfowler.com](https://martinfowler.com/articles/microservices.html)

### Дополнительная

6.  Newman, S. (2021). *Building Microservices* (2nd ed.). O'Reilly. — [O'Reilly](https://www.oreilly.com/library/view/building-microservices-2nd/9781492034018/) | [Amazon](https://www.amazon.com/Building-Microservices-Designing-Fine-Grained-Systems/dp/1492034029)
7.  Richards, M. (2015). *Software Architecture Patterns*. O'Reilly. — [O'Reilly (бесплатный PDF после регистрации)](https://www.oreilly.com/library/view/software-architecture-patterns/9781491971437/)
8.  Evans, E. (2003). *Domain-Driven Design: Tackling Complexity in the Heart of Software*. Addison-Wesley. — [Официальная страница](https://domainlanguage.com/ddd/) | [Amazon](https://www.amazon.com/Domain-Driven-Design-Tackling-Complexity-Software/dp/0321125215)
9.  Nygard, M., et al. (2011–н.в.). *Architecture Decision Records*. — [ADR GitHub (сообщество)](https://adr.github.io/) | [ADR Tools](https://github.com/npryce/adr-tools) | [Статья М. Найгарда](https://cognitect.com/blog/2011/11/15/documenting-architecture-decisions)
10. Hohpe, G., & Woolf, B. (2003). *Enterprise Integration Patterns*. Addison-Wesley. — [Официальная страница](https://www.enterpriseintegrationpatterns.com/) — фундаментальная книга по SOA и обмену сообщениями.
11. Stopford, B. (2018). *Designing Event-Driven Systems*. O'Reilly. — [Бесплатный PDF от Confluent](https://www.confluent.io/designing-event-driven-systems/) — практическое руководство по EDA на Kafka.
12. Kleppmann, M. (2017). *Designing Data-Intensive Applications*. O'Reilly. — [O'Reilly](https://www.oreilly.com/library/view/designing-data-intensive-applications/9781491903063/) — главы про event sourcing, CQRS, streaming.

### Литература к разделу 3.6 (NFR и архитектура)

13. Bass, L., Clements, P., & Kazman, R. (2021). *Software Architecture in Practice* (4th ed.). Addison-Wesley. — [O'Reilly](https://www.oreilly.com/library/view/software-architecture-in/9780136885962/) | [Amazon](https://www.amazon.com/Software-Architecture-Practice-4th/dp/0136886094) — каноничная книга по Quality Attribute Scenarios и ATAM.
14. ISO/IEC 25010:2023. *Systems and software Quality Requirements and Evaluation (SQuaRE) — Product quality model*. — [Официальная страница ISO](https://www.iso.org/standard/78176.html)
15. Kazman, R., Klein, M., & Clements, P. (2000). *ATAM: Method for Architecture Evaluation*. SEI, Carnegie Mellon. — [PDF (бесплатно)](https://insights.sei.cmu.edu/library/atam-method-for-architecture-evaluation/)
16. Richards, M., & Ford, N. (2020). *Fundamentals of Software Architecture*. O'Reilly. — [O'Reilly](https://www.oreilly.com/library/view/fundamentals-of-software/9781492043454/) — главы про architecture characteristics (NFR) и выбор стиля.
17. Chen, L., & Babar, M. A. (2011). *Towards an Evidence-Based Understanding of Electronic Data Sources for Architecture Evaluation*. — Научная статья про методы оценки архитектуры.

### Онлайн-ресурсы

18. Martin Fowler's Blog — [https://martinfowler.com/](https://martinfowler.com/)
19. Microsoft Azure Architecture Center — [https://learn.microsoft.com/en-us/azure/architecture/](https://learn.microsoft.com/en-us/azure/architecture/)
20. Awesome Software Architecture (GitHub) — [https://github.com/mehdihadeli/awesome-software-architecture](https://github.com/mehdihadeli/awesome-software-architecture)
21. Refactoring.Guru — [https://refactoring.guru/](https://refactoring.guru/) — визуальные объяснения SOLID, паттернов и рефакторинга (RU/EN).
22. Microservices.io (Chris Richardson) — [https://microservices.io/](https://microservices.io/) — паттерны микросервисной архитектуры.
23. Enterprise Integration Patterns — [https://www.enterpriseintegrationpatterns.com/](https://www.enterpriseintegrationpatterns.com/) — каталог паттернов обмена сообщениями (актуально для SOA и EDA).
24. Confluent Blog — [https://www.confluent.io/blog/](https://www.confluent.io/blog/) — статьи по event streaming и Kafka.
25. SEI (Software Engineering Institute) — [https://www.sei.cmu.edu/](https://www.sei.cmu.edu/) — материалы по ATAM и архитектурной оценке.